# 1D Diffusion

In the previous two lessons, we studied the numerical solution of the linear and nonlinear convection equations and learned about the CFL stability condition. We now turn to the **one-dimensional diffusion equation**:

$$
\label{eq-1d-diffusion-pde}
\frac{\partial u}{\partial t}=\nu\frac{\partial^2 u}{\partial x^2}.
$$

Here, $\nu>0$ is a constant *diffusion coefficient*, with units of length squared over time. This equation describes how spatial variations in $u$ smooth out over time. It applies to several physical settings, like:

- the evolution of temperature along a rod, where it is called the *heat equation*;
- the diffusion of a pollutant along a pipe containing still fluid; or
- the diffusion of a dissolved substance through a stationary gel.

On a finite domain, its solution needs an initial profile and a boundary condition at each end. In this lesson, we use $0\le x\le L$, with $L=2$, and hold both endpoint values fixed:

$$
\label{eq-1d-diffusion-boundaries}
u(0,t)=u(L,t)=1,\qquad t\ge0.
$$

These prescribed values are called *Dirichlet boundary conditions*. We will specify the initial profile below.

This is the first instance where we encounter a **second-order spatial derivative**. Our first-derivative schemes do not directly provide an approximation for this term, so we need to derive a new finite-difference formula before advancing the solution in time.

## Discretize the second derivative

The second spatial derivative measures how the *slope* of $u$ changes with position. To approximate it, we use the value at a grid point and its two immediate neighbors.

As in [Lesson 6](./06-1d-convection.ipynb), take a uniform grid $x_i=i\Delta x$, with $i=0,\ldots,N$ and $\Delta x=L/N$. At a fixed time, write $u_i=u(x_i,t)$ for the exact values while deriving the spatial approximation. For a sufficiently smooth function, Taylor expansion about $x_i$ gives

$$
\label{eq-diffusion-taylor-pair}
\begin{aligned}
u_{i+1} &= u_i+\Delta x\left.\frac{\partial u}{\partial x}\right|_i
+\frac{\Delta x^2}{2}\left.\frac{\partial^2 u}{\partial x^2}\right|_i
+\frac{\Delta x^3}{6}\left.\frac{\partial^3 u}{\partial x^3}\right|_i
+\mathcal{O}(\Delta x^4),\\
u_{i-1} &= u_i-\Delta x\left.\frac{\partial u}{\partial x}\right|_i
+\frac{\Delta x^2}{2}\left.\frac{\partial^2 u}{\partial x^2}\right|_i
-\frac{\Delta x^3}{6}\left.\frac{\partial^3 u}{\partial x^3}\right|_i
+\mathcal{O}(\Delta x^4).
\end{aligned}
$$

Adding the expansions cancels the displayed odd-derivative terms and gives:

$$
u_{i+1}+u_{i-1}=2u_i+\Delta x^2\left.\frac{\partial^2 u}{\partial x^2}\right|_i
+\mathcal{O}(\Delta x^4).
$$

Subtract $2u_i$ and divide by $\Delta x^2$ to obtain

$$
\label{eq-diffusion-centered-second-derivative}
\left.\frac{\partial^2 u}{\partial x^2}\right|_i
=\frac{u_{i+1}-2u_i+u_{i-1}}{\Delta x^2}
+\mathcal{O}(\Delta x^2).
$$

This is a **second-order accurate central difference**: for a smooth profile, its leading error scales with $\Delta x^2$. Notice that dividing by $\Delta x^2$ also changes the order of the remainder from $\mathcal{O}(\Delta x^4)$ to $\mathcal{O}(\Delta x^2)$.

:::{warning .simple .dropdown icon=false open=false} On paper
Reconstruct the derivation by adding the two Taylor expansions. Then consider the numerator $u_{i+1}-2u_i+u_{i-1}$: what sign does it have at a strict local maximum? What value does it have when the three points lie on a straight line? Use the diffusion equation to predict the corresponding change in $u_i$.
:::

## Forward Euler in time

Let $u_i^n$ denote our numerical approximation to $u(x_i,t^n)$, where $t^n=n\Delta t$. Using the central difference in [Equation %s](#eq-diffusion-centered-second-derivative) for the spatial derivative, we apply **forward Euler** to advance in time. As in our earlier ODE calculations, Euler's method evaluates the rate of change using the current state:

$$
\label{eq-diffusion-ftcs}
\frac{u_i^{n+1}-u_i^n}{\Delta t}
=\nu\frac{u_{i+1}^n-2u_i^n+u_{i-1}^n}{\Delta x^2}.
$$

Rearranging gives the explicit update

$$
\label{eq-diffusion-explicit-update}
u_i^{n+1}=u_i^n+\frac{\nu\Delta t}{\Delta x^2}
\left(u_{i+1}^n-2u_i^n+u_{i-1}^n\right),
\qquad i=1,\ldots,N-1.
$$

Every value on the right-hand side belongs to the old time level $n$. The new interior values can therefore be calculated directly from the old array. This combination is called **forward-time, central-space (FTCS)**. It is first-order accurate in time and second-order accurate in space; these orders describe the truncation error for sufficiently smooth solutions.

The three-point formula applies to the interior points. At the two endpoints, we impose the prescribed boundary values instead:

$$
u_0^{n+1}=u_N^{n+1}=1.
$$

For the initial profile, we reuse the square pulse from the convection lesson:

$$
\label{eq-diffusion-initial-profile}
u(x,0)=
\begin{cases}
2, & 0.5\le x\le1,\\
1, & \text{elsewhere on }[0,2].
\end{cases}
$$

This profile makes the spreading of the pulse easy to see. Its jumps do not satisfy the smoothness assumption used in the Taylor derivation, however, so we should not use them to infer the formal order of accuracy.

:::{warning .simple .dropdown icon=false open=false} On paper
For the three values $[1,2,1]$ and $\nu\Delta t/\Delta x^2=0.2$, calculate the updated middle value while holding the endpoints fixed. Does the change agree with your prediction at a local maximum?
:::

## Stability through convex weights

In [Lesson 7](./07-cfl-condition.ipynb), we established a stability bound by writing the convection update as a weighted average. We can reuse that argument for diffusion, now with three old values instead of two.

### Read the weights

Define the **diffusion number**

$$
\label{eq-diffusion-number}
r=\frac{\nu\Delta t}{\Delta x^2}.
$$

This ratio is dimensionless: $\nu$ has units of length squared per time. Collecting terms in [Equation %s](#eq-diffusion-explicit-update) gives

$$
\label{eq-diffusion-convex-update}
u_i^{n+1}=r u_{i-1}^n+(1-2r)u_i^n+r u_{i+1}^n.
$$

The weights sum to one. They are all nonnegative when

$$
\label{eq-diffusion-convex-condition}
0\le r\le\frac12.
$$

Under this condition, the new value is a *convex combination*: it lies between the smallest and largest of the three old values. With our fixed endpoint values, the update cannot create a value outside the range of the initial and boundary data. For the square pulse, that range is $[1,2]$.

This explains a useful property of the solution, but stability asks about the amplification of a disturbance. As in Lesson 7, we apply the argument to the difference between two calculations.

### Bound a disturbance

Consider two numerical solutions with the same grid, diffusivity, time step, and prescribed endpoint values, but slightly different initial profiles. Define

$$
\label{eq-diffusion-perturbation}
e_i^n=u_{b,i}^n-u_{a,i}^n,
\qquad E^n=\max_i|e_i^n|.
$$

Here, $e_i^n$ is the difference between two numerical solutions, not the error relative to an exact solution. Subtracting their updates gives

$$
\label{eq-diffusion-perturbation-update}
e_i^{n+1}=r e_{i-1}^n+(1-2r)e_i^n+r e_{i+1}^n.
$$

For $0\le r\le1/2$, the triangle inequality and the nonnegative weights imply

$$
\label{eq-diffusion-perturbation-bound}
\begin{aligned}
|e_i^{n+1}|
&\le r|e_{i-1}^n|+(1-2r)|e_i^n|+r|e_{i+1}^n|\\
&\le rE^n+(1-2r)E^n+rE^n\\
&=E^n.
\end{aligned}
$$

Both endpoint differences are zero because the prescribed boundary values are identical. Taking the maximum over the whole array and repeating the bound over successive steps therefore yields

$$
\label{eq-diffusion-stability-bound}
E^n\le E^{n-1}\le\cdots\le E^0.
$$

The amplification bound is one, independent of the grid spacing and number of steps. This establishes stability for the stated update and boundary treatment when $0\le r\le1/2$. It does not establish that the numerical solution is sufficiently accurate.

### What happens beyond the limit?

When $r>1/2$, the middle weight is negative. The argument above no longer applies, but a failed proof is not itself proof of growth. To see a mechanism for growth, reuse the alternating disturbance from Lesson 7. On a grid without boundaries, let

$$
e_i^n=A_n(-1)^i.
$$

Both neighbors of the point $i$ have the opposite sign. Substitution into [Equation %s](#eq-diffusion-perturbation-update) gives

$$
\label{eq-diffusion-alternating-factor}
A_{n+1}=(1-4r)A_n.
$$

For $r>1/2$, the factor $1-4r$ is less than $-1$: the disturbance reverses sign and grows in magnitude at every step. For example, $r=0.6$ gives a factor of $-1.4$. At $r=1/2$, the factor is $-1$, so this particular pattern changes sign without decaying. Stability permits that behavior; it does not require every disturbance to decay.

Our two solutions have identical fixed endpoint values, so their difference is zero at both ends. The alternating disturbance therefore stops at the boundaries. Each update uses only neighboring values, so the boundary’s effect moves inward by at most one grid point per step. For example, over five steps, the factor $1-4r$ predicts the disturbance exactly at points more than five grid intervals from either end. Closer to the endpoints, the disturbance no longer follows this simple alternating pattern.

### The time step now scales with the square of the spacing

For positive diffusivity, the bound gives

$$
\label{eq-diffusion-time-step-limit}
\Delta t\le\frac{\Delta x^2}{2\nu}.
$$

Halving $\Delta x$ therefore quarters the largest permitted time step. Keeping the same diffusion number on the finer grid requires four times as many steps to reach the same physical time. In contrast, the convection time-step limit in Lesson 7 scaled with $\Delta x$. The diffusion number measures a different balance from the distance-per-step interpretation of the Courant number.

:::{warning .simple .dropdown icon=false open=false} On paper
1. Reconstruct the bound on $E^{n+1}$, identifying where nonnegative weights and identical boundary data enter the argument.
2. Calculate the three weights and the alternating-disturbance factor for $r=0.2$, $0.5$, and $0.6$. Which cases control the magnitude of that disturbance?
3. If you halve $\Delta x$ while keeping $\Delta t$ fixed, what happens to $r$? Starting from $r=0.2$, does the refined calculation still satisfy the bound?
:::

## Compute the diffusion of a square pulse

We can now implement [Equation %s](#eq-diffusion-explicit-update). We use the same domain and square pulse as above, with $\nu=0.3$ and a target diffusion number of $0.2$, within the bound we just derived.

As in Lesson 7, we choose a final time first. The target diffusion number gives a proposed time step; we round the number of steps up and adjust the step size to reach that time. The realized diffusion number may be slightly smaller than the target. This lets us change the grid without also changing the duration of the calculation.

:::{warning .simple .dropdown icon=false open=false} In your notebook
Reconstruct the grid, initial profile, and time loop below. Before running the calculation, predict how the pulse will change and what range its values should remain in. You may copy the imports and plot formatting.
:::

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

In [ ]:
# Define the grid and diffusion coefficient
nx = 41
L = 2.0
dx = L / (nx - 1)
nu = 0.3
x = np.linspace(0.0, L, num=nx)

# Choose a time step that reaches the requested final time
t_final = 0.04
r_target = 0.2
dt_target = r_target * dx**2 / nu

num_steps = int(np.ceil(t_final / dt_target))
dt = t_final / num_steps
r = nu * dt / dx**2

# Set the initial profile, including the fixed endpoint values
u0 = np.ones(nx)
mask = (x >= 0.5) & (x <= 1.0)
u0[mask] = 2.0

print(f'dx = {dx:.4f}, dt = {dt:.6f}, r = {r:.3f}')
print(f'{num_steps} steps to t = {num_steps * dt:.4f}')

In the initial-condition setup, the Boolean mask selects points satisfying both comparisons. The `&` combines the two Boolean arrays element by element; keep the parentheses around each comparison.

Now you can perform the updates using only a time loop, and array operations for the spatial stencil.

In [ ]:
# Advance only the interior points; both endpoints remain fixed at 1.
u = u0.copy()
for n in range(num_steps):
    u[1:-1] = u[1:-1] + r * (u[2:] - 2 * u[1:-1] + u[:-2])

:::{note} Python refresher — aligned slices
:icon: false
A slice includes its starting index and excludes its stopping index. Negative indices count backward from the end, so `-1` identifies the last element. The three slices below have the same length and align each interior point with its neighbors:

| Slice | Grid values | Role |
| --- | --- | --- |
| `u[:-2]` | $u_0,\ldots,u_{N-2}$ | Left neighbors |
| `u[1:-1]` | $u_1,\ldots,u_{N-1}$ | Interior points |
| `u[2:]` | $u_2,\ldots,u_N$ | Right neighbors |

For a five-element array, these slices select indices `[0, 1, 2]`, `[1, 2, 3]`, and `[2, 3, 4]`, respectively. NumPy performs the arithmetic element by element, forming all the three-point differences at once.

The entire right-hand side is evaluated before assignment to `u[1:-1]`, so every difference uses the old time level. A left-to-right loop that overwrites individual values could instead mix old and new values. The endpoint entries are never assigned here, so they retain their prescribed values. Finally, `u0.copy()` makes an independent working array, preserving the initial profile for comparison.
:::

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 3.0))
ax.plot(x, u0, label='Initial', linestyle='--', linewidth=2)
ax.plot(x, u, color='tab:red', label=f't = {t_final:.3f}', linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('u')
ax.set_xlim(0.0, L)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout();

The pulse spreads and its peak decreases, while both endpoints stay at $1$. Unlike the pure-convection example, smoothing is part of the physical model here. The convex-weight argument predicts that the values remain within $[1,2]$; the plot alone does not establish that the amount of diffusion is accurate.

:::{warning .simple .dropdown icon=false open=false} Self-checks
- Confirm that `u[0]` and `u[-1]` are still $1$, and inspect `u.min()` and `u.max()` against your predicted range.
- Why is `u = u0.copy()` needed if we want to plot the initial and final profiles together?
- Change `nx` from $41$ to $81$ and rerun the calculation starting from the parameter cell. Compare the reported time step and step count. Do they agree with the scaling derived above, and do both calculations reach the same final time?
:::

## Assessing accuracy

The square pulse spreads and its peak decreases, as we expect from diffusion. But a stable calculation with a plausible shape could still diffuse too quickly or too slowly. **How can we check that the amount of diffusion is correct?**

We need a reference whose evolution we know independently of the numerical calculation. For this purpose, replace the square pulse with a smooth sine profile on the same domain, keeping the endpoint values fixed at $1$:

$$
\label{eq-diffusion-sine-initial}
u(x,0)=1+A\sin\left(\frac{\pi x}{L}\right).
$$

We will use $A=1$, so the initial peak is $2$, as it was for the square pulse. On $0\le x\le L$, the sine is nonnegative, so the profile lies within $[1,2]$.

### A solution we can check on paper

For this initial profile, a candidate exact solution is

$$
\label{eq-diffusion-sine-exact}
u(x,t)=1+A e^{-\nu k^2t}\sin(kx),
\qquad k=\frac{\pi}{L}.
$$

The symbol $k$ keeps the expressions short. It has units of inverse length, making both $kx$ and $\nu k^2t$ dimensionless. The sine determines the spatial shape, while the exponential determines how its amplitude changes with time.

Before using this expression to assess our code, we should verify that it satisfies the equation and the initial and boundary conditions.

:::{warning .simple .dropdown icon=false open=false} On paper — check the candidate solution
1. Differentiate $u$ with respect to $t$, holding $x$ fixed. Treat $A\sin(kx)$ as a constant multiplier.
2. Differentiate $u$ twice with respect to $x$, holding $t$ fixed. This time, $A e^{-\nu k^2t}$ is the constant multiplier.
3. Compare your expressions for $u_t$ and $\nu u_{xx}$. Are they equal?
4. Set $t=0$. Do you recover [Equation %s](#eq-diffusion-sine-initial)? Then set $x=0$ and $x=L$. Do the endpoint values remain $1$ for all time?

**Calculus hints:** the derivative of the constant background $1$ is zero. Use the chain rule:

$$
\frac{d}{dt}e^{-\nu k^2t}=-\nu k^2e^{-\nu k^2t},
\qquad
\frac{d}{dx}\sin(kx)=k\cos(kx),
\qquad
\frac{d}{dx}\cos(kx)=-k\sin(kx).
$$

Each spatial differentiation contributes a factor of $k$. For the last check, recall $e^0=1$ and $\sin(0)=\sin(\pi)=0$.

After taking the derivatives, you should obtain

$$
\label{eq-diffusion-sine-derivatives}
\begin{aligned}
u_t&=-\nu k^2 A e^{-\nu k^2t}\sin(kx),\\
u_{xx}&=-k^2 A e^{-\nu k^2t}\sin(kx).
\end{aligned}
$$

Multiplying the second expression by $\nu$ gives the first. At $t=0$, the exponential equals one; at either endpoint, the sine equals zero. The expression therefore satisfies the full problem we specified, not just the PDE.
:::

### Predict the decay

The exact solution retains its sine shape while the amplitude above the background decreases as

$$
\label{eq-diffusion-sine-amplitude}
a(t)=A e^{-\nu k^2t}.
$$

At the midpoint $x=L/2$, the sine equals one, so $u(L/2,t)-1=a(t)$. This gives us a simple quantity to measure in the numerical solution.

With $L=2$ and $\nu=0.3$, the decay rate $\nu k^2$ is approximately $0.7402$. Over the short interval $T=0.04$ used above, the amplitude decreases by only about $2.9\%$. For the accuracy investigation, we will use **$T=1$**: the amplitude then decreases from $1$ to approximately $0.4770$, and the peak value becomes approximately $1.4770$.

These predictions come from the exact solution, before we run FTCS. A numerical peak below $2$ would show decay, but would not tell us whether the rate is correct.

### Predict the numerical solution

For this particular profile, we can also predict the numerical solution without running the time loop. This gives us a second reference: one for checking the implementation of the discrete update.

Subtract the constant background by defining $v_i^n=u_i^n-1$. The constant terms cancel in the spatial difference, so $v$ follows the same FTCS update, with zero values at both endpoints. Suppose its profile at step $n$ is

$$
v_i^n=a_n\sin(kx_i).
$$

To substitute this into the stencil, we need the neighboring sine values. Since $x_{i\pm1}=x_i\pm\Delta x$, the angle-addition identity gives

$$
\label{eq-diffusion-sine-neighbors}
\sin(kx_{i+1})+\sin(kx_{i-1})
=2\sin(kx_i)\cos(k\Delta x).
$$

Using this identity in the FTCS update yields

$$
\begin{aligned}
v_i^{n+1}
&=a_n\sin(kx_i)
+r a_n\left[2\sin(kx_i)\cos(k\Delta x)-2\sin(kx_i)\right]\\
&=a_n\left[1+2r\bigl(\cos(k\Delta x)-1\bigr)\right]\sin(kx_i).
\end{aligned}
$$

Recall that $\cos\theta-1=-2\sin^2(\theta/2)$. The amplitude therefore obeys

$$
\label{eq-diffusion-sine-step-factor}
a_{n+1}=\lambda a_n,
\qquad
\lambda=1-4r\sin^2\left(\frac{k\Delta x}{2}\right).
$$

The sampled sine keeps its shape: each time step multiplies its amplitude by $\lambda$. Starting from $a_0=A$, repeated multiplication gives the exact solution of the discrete scheme:

$$
\label{eq-diffusion-sine-discrete}
u_{i,\mathrm{discrete}}^n
=1+A\lambda^n\sin(kx_i).
$$

The sine vanishes at both endpoints, so this expression also satisfies the discrete boundary conditions. Here, “exact” means exact arithmetic for the finite-difference scheme; it does not mean that the scheme reproduces the PDE solution exactly.

:::{warning .simple .dropdown icon=false open=false} On paper — follow one numerical step
Use [Equation %s](#eq-diffusion-sine-neighbors) to reconstruct the substitution into FTCS. Identify the factor multiplying $a_n$, then write the amplitudes after one, two, and three steps. Why does multiplying the sine by this factor preserve the endpoint values?
:::

This is the same strategy we used for the alternating disturbance: choose a spatial pattern and determine how the update changes its amplitude. That earlier interior calculation gave the factor $1-4r$. Here we have used a smooth sine that satisfies our fixed endpoint conditions.

### Two references, two questions

At time $t^n=n\Delta t$, we can now distinguish:

- **Implementation:** does the computed array agree, up to roundoff errors, with $1+A\lambda^n\sin(kx_i)$? This checks whether our code produces the expected FTCS result for this case.
- **Discretization:** how far does that discrete solution differ from $1+A e^{-\nu k^2t^n}\sin(kx_i)$? This measures the error introduced by approximating the PDE with FTCS.

Both reference profiles have the same sampled sine shape. Their difference is entirely in amplitude:

$$
\label{eq-diffusion-sine-amplitude-error}
u_{i,\mathrm{discrete}}^n-u(x_i,t^n)
=A\left(\lambda^n-e^{-\nu k^2t^n}\right)\sin(kx_i).
$$

Agreement with the discrete reference supports the implementation for this test case. Agreement with the continuous reference addresses accuracy for the specified diffusion problem. The next calculation will use both checks; a plausible plot alone cannot replace either one.

### Compute and compare the decay

We now reuse the FTCS update from the square-pulse calculation, changing the initial profile and extending the final time to $T=1$. The grid still contains $41$ points, including $x=L/2$. We will record the amplitude there after each step, rather than store every full solution array.

:::{warning .simple .dropdown icon=false open=false} In your notebook
Reconstruct the sine-profile setup and reuse your FTCS time loop. Before running it, record the exact amplitude predicted at $T=1$. Add an array to record $u(L/2,t)-1$ after each update. Keep the numerical evolution separate from the reference formulas: the time loop must calculate the solution using the stencil, not the known amplitude factor.
:::

In [ ]:
# Set up the sine-profile problem on the same domain
nx = 41
L  = 2.0
nu = 0.3
A  = 1.0
k  = np.pi / L
dx = L / (nx - 1)
x  = np.linspace(0.0, L, num=nx)

t_final = 1.0
r_target = 0.2
dt_target = r_target * dx**2 / nu
num_steps = int(np.ceil(t_final / dt_target))
dt = t_final / num_steps
r = nu * dt / dx**2

sine_profile = np.sin(k * x)
sine_profile[0] = sine_profile[-1] = 0.0
u0 = 1.0 + A * sine_profile

print(f'dx = {dx:.4f}, dt = {dt:.8f}, r = {r:.4f}')
print(f'{num_steps} steps to t = {num_steps * dt:.4f}')

Setting the two endpoint entries of `sine_profile` to zero imposes the boundary values exactly, without relying on floating-point evaluation of $\sin(\pi)$.

The expression `nx // 2` gives the midpoint index for this odd number of equally spaced grid points. The amplitude history has `num_steps + 1` entries: one for the initial state and one for each completed update. The time array uses the same indexing.

In [ ]:
midpoint = nx // 2
time = dt * np.arange(num_steps + 1)
amplitude = np.empty(num_steps + 1)

u = u0.copy()
amplitude[0] = u[midpoint] - 1.0
for n in range(num_steps):
    u[1:-1] = u[1:-1] + r * (u[2:] - 2 * u[1:-1] + u[:-2])
    amplitude[n + 1] = u[midpoint] - 1.0

Now evaluate the two independent predictions at those same times. For the discrete reference, write $\lambda=1-q$, where $q=4r\sin^2(k\Delta x/2)$. Then $\lambda^n=\exp[n\log(1-q)]$.

We use `np.log1p(-q)`, which evaluates $\log(1-q)$ accurately when $q$ is small. This avoids losing precision by first subtracting a small number from $1$. For the sine mode and parameters used here, $0<\lambda<1$, so its logarithm is defined.

In [ ]:
exact_rate = nu * k**2
amplitude_exact = A * np.exp(-exact_rate * time)

q = 4 * r * np.sin(k * dx / 2)**2
log_factor = np.log1p(-q)
amplitude_discrete = A * np.exp(np.arange(num_steps + 1) * log_factor)

u_exact = 1.0 + amplitude_exact[-1] * sine_profile
u_discrete = 1.0 + amplitude_discrete[-1] * sine_profile

An exponential decay appears as a straight line when amplitude is plotted on a logarithmic vertical axis: $\log a(t)=\log A-\nu k^2t$. We plot the computed amplitude at selected times to keep the three comparisons readable. All values were computed and recorded; only the displayed markers are thinned.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 3.0))
ax.semilogy(time, amplitude_exact, color='black', label='Exact PDE')
ax.semilogy(time, amplitude_discrete, color='tab:blue',
            linestyle='--', label='Discrete prediction')
ax.semilogy(time[::30], amplitude[::30], color='tab:red',
            marker='o', markersize=3, linestyle='none', label='Computed FTCS')
ax.set_xlabel('Time')
ax.set_ylabel('Amplitude above background')
ax.grid(which='both', alpha=0.3)
ax.legend()
fig.tight_layout();

The three curves nearly coincide at this plotting scale. We need numbers to resolve their differences. For an exponential, the decay rate can be recovered from its amplitude change over the interval:

$$
\label{eq-diffusion-measured-decay-rate}
\beta_{\mathrm{measured}}
=-\frac{1}{T}\log\left(\frac{a(T)}{a(0)}\right).
$$

The PDE predicts $\beta=\nu k^2$, while the discrete formula predicts $\beta_{\mathrm{discrete}}=-\log(\lambda)/\Delta t$. For this single sine mode, each has a constant decay rate. For a general initial profile, an endpoint amplitude ratio would not establish exponential decay.

In [ ]:
measured_rate = -np.log(amplitude[-1] / amplitude[0]) / time[-1]
discrete_rate = -log_factor / dt

print(f"{'Reference':<22} {'Final amplitude':>18} {'Decay rate':>16}")
for name, final_amplitude, rate in (
    ('Exact PDE', amplitude_exact[-1], exact_rate),
    ('Discrete prediction', amplitude_discrete[-1], discrete_rate),
    ('Computed FTCS', amplitude[-1], measured_rate),
):
    print(f'{name:<22} {final_amplitude:18.10f} {rate:16.10f}')

The exact final amplitude is approximately $0.4770088$, while FTCS gives approximately $0.4769725$. Its decay rate is approximately $0.7402964$, slightly larger than the PDE rate $0.7402203$. On this grid with this time step, the numerical solution decays slightly too quickly.

### Separate implementation discrepancy from discretization error

Measure the differences over the entire final array using the maximum absolute value, the same measure of size used in our stability argument:

$$
\label{eq-diffusion-two-reference-errors}
\begin{aligned}
R_{\mathrm{implementation}}
&=\max_i\left|u_i^{\mathrm{computed}}-u_{i,\mathrm{discrete}}\right|,\\
E_{\mathrm{discretization}}
&=\max_i\left|u_{i,\mathrm{discrete}}-u(x_i,T)\right|,\\
E_{\mathrm{total}}
&=\max_i\left|u_i^{\mathrm{computed}}-u(x_i,T)\right|.
\end{aligned}
$$

The **implementation discrepancy** compares the code with the derived result of the scheme. The **discretization error** compares the scheme with the PDE. The **total error** compares what we actually computed with the PDE. These are different comparisons; in general, their scalar values do not simply add, although the triangle inequality gives $E_{\mathrm{total}}\le R_{\mathrm{implementation}}+E_{\mathrm{discretization}}$.

In [ ]:
implementation_discrepancy = np.max(np.abs(u - u_discrete))
discretization_error = np.max(np.abs(u_discrete - u_exact))
total_error = np.max(np.abs(u - u_exact))

print(f'Implementation discrepancy: {implementation_discrepancy:.3e}')
print(f'Discretization error:       {discretization_error:.3e}')
print(f'Total error against PDE:    {total_error:.3e}')

The implementation discrepancy should be near floating-point roundoff, many orders of magnitude below the discretization error of approximately $3.63\times10^{-5}$. Its precise value can vary with floating-point evaluation. The total error is consequently almost entirely explained by discretization in this run.

This is evidence that our code implements FTCS correctly **for this benchmark**. It does not make FTCS exact: the predicted discrete amplitude differs from the exact PDE amplitude. Conversely, a substantial disagreement with the discrete reference would require us to inspect the implementation, parameters, and reference calculation before attributing the discrepancy to the method.

:::{warning .simple .dropdown icon=false open=false} Self-checks
- Why do we subtract $1$ before measuring amplitude? What would go wrong with using the peak value itself in the decay-rate formula?
- Which comparison supports the implementation, and which measures discretization error? Explain why a small implementation discrepancy can coexist with a larger total error.
- Does FTCS decay too quickly or too slowly in this run? Support your answer with both the final amplitudes and the decay rates.
- The curves look coincident. Which numerical evidence shows that they are not identical, and what remains unknown about the benefit of refining the grid?
:::